# Очистка текстов
### Удаляем обозначения говорящих/поющих


In [3]:
import re
import os
import pandas as pd

### Импорт данных

In [3]:
songs = pd.read_excel('libretto_en_ru.xlsx')

In [4]:
songs.head()

,musical,songtitle_eng,text_eng,songtitle_rus,text_rus
0,Mamma Mia!,"Honey, Honey","DONNA\nHoney honey, how he thrills me, a-ha, h...","МИЛЫЙ, МИЛЫЙ","Милый, Милый,\nОн волнует меня\nМилый, Милый.\..."
1,Mamma Mia!,"Money, Money, Money","DONNA\nI work all night, I work all day\nTo pa...","ДЕНЬГИ, ДЕНЬГИ, ДЕНЬГИ","Работай день, работай ночь, чтоб как-то выкрут..."
2,Mamma Mia!,Thank You for the Music,HARRY\nThank you for the music\nThe songs I'm ...,СПАСИБО ТЕБЕ ЗА МУЗЫКУ,"Я скажу спасибо тебе за песни,\nИх мелодий хор..."
3,Mamma Mia!,Mamma Mia,DONNA\nI was cheated by you\nAnd I think you k...,МАМА МИЯ,"Ты меня обманул, сам ты помнишь когда?\nЯ реши..."
4,Mamma Mia!,Chiquitita,"ROSIE\nChiquitita, tell me what's wrong\nTANYA...",ЧИКИТИТА,"Чикитита, что же с тобой?\nЭто горе так внезап..."


In [ ]:
# на всякий случай удаляем строки, если в них отсутствует один из текстов
songs_filtered = songs.dropna(subset=['text_eng', 'text_rus'])

In [182]:
mustitles = songs_filtered['musical'].unique()
mustitles

array(['Mamma Mia!', 'Шахматы', 'Джекилл и Хайд', 'Кошки', 'Волосы'],
      dtype=object)

### Удаление обозначений, кому принадлежат строки в песнях
Пометки в духе:

ПЕРСОНАЖ: ; [ВСЕ ВМЕСТЕ] ; Персонаж А, персонаж Б и персонаж В: ; ПЕРСОНАЖ (с тревогой): и т.д.

In [56]:
def get_speakers_by_song(song, pattern):
    ''' Находит в текстах песен отметки о том, какие персонажи поют тот или иной фрагмент. 

        Для сложных случаев возможен ввод иного паттерна для поиска спикеров:
        pattern = 'default' - стандартный паттерн (предполагается, что все спикеры в верхнем регистре и строчных букв нет в них, 
                                                    только м.б. как дополнение к спикеру - spoken, crying и т.д.)
        pattern = 'expanded' - расширенный стандартный паттерн (возможен нижний регистр, но точно есть двоеточие в конце)
        pattern = 'custom' - ввод кастомного паттерна с клавиатуры
        '''
    text = song
    
    if pattern == 'default':
        chosen_regex = r'^[А-ЯЁA-Z- ‒–—―\,\'\&\(\)\\\/\d]+(?: )?(?:\([a-zA-Zа-яёА-ЯЁ -]+?\))?\:?(?: )*?$'
    elif pattern == 'expanded':
        chosen_regex = r'^[А-ЯЁа-яёA-Za-z-‒–—―\&\(\)\\\/\d\,]+(?:[ \(]+)?[А-ЯЁа-яёA-Za-z-‒–—―\(\)\\\/\d\,]*?(?: )?(?:[А-ЯЁA-Z\d][а-яёa-z-‒–—―\(\)\\\/\d\,]+){0,4}(?:[ \)]+)?\:(?: )*?$'
    else:
        input_pattern = pattern
        chosen_regex = fr'{input_pattern}'
    

    speakers = set(re.findall(chosen_regex, text, flags=re.MULTILINE))
    
    # сортируем по длине в порядке убывания, чтобы не было ситуации, когда в строке типа "Персонаж1 и Персонаж2:" 
    # регулярка поймала только "Персонаж2:", который как отдельный элемент тоже есть, и получится "Персонаж1 и ПЕРСОНАЖ2:" (нам нужно капсом всё)
    speakers_fine = sorted(speakers, key=len, reverse = True)
    # speakers_suspicious = [el for el in speakers_fine if len(el) > 25]
    
    return speakers_fine


def speakers_deleting(string, speakers_set):
    ''' Удаление авторов реплик из общего заданного списка в конкретной строке '''
    edited = string
    if speakers_set != []:
        for s in speakers_set:
            if s in string:
                edited = re.sub(re.escape(s), '', edited)
    else:
        pass
    

    return edited

In [ ]:
# общая функция для удаления спикеров с опорой на предыдущие
def text_cleaning(song_string, speakers_pattern):
    ''' Удаляет спикеров из текста песни, если они присутствуют '''

    string_copy = song_string
    speakers = get_speakers_by_song(song = song_string, pattern = speakers_pattern)
    if speakers != []:
        for s in speakers:
            string_copy = speakers_deleting(string = string_copy, speakers_set = speakers)
    else:
        pass

    return string_copy, speakers

# то же самое, но возвращает только отредактированную строку
def text_cleaning_stringonly(song_string, speakers_pattern):
    ''' Удаляет спикеров из текста песни, если они присутствуют '''

    string_copy = song_string
    speakers = get_speakers_by_song(song = song_string, pattern = speakers_pattern)
    if speakers != []:
        for s in speakers:
            string_copy = speakers_deleting(string = string_copy, speakers_set = speakers)


    return string_copy

Прикинем, какой из паттернов для поиска и удаления обозначений говорящих/поющих будет лучше в каждом конкретном случае на обоих языках. В каких-то текстах спикеров нет в принципе - там, соответственно, поиск с speakers_pattern = 'default' и = 'expanded' выдаст просто пустой список.

In [11]:
# посмотрим на то, как отлавливает спикеров в англ. текстах дефолтный паттерн
speakers_dict = {}
for mus in mustitles:
    subset = songs_filtered[songs_filtered['musical'] == mus]
    # print(mus, len(subset))
    found_default = []
    found_expanded = []
    for i in subset['text_eng'].apply(text_cleaning, speakers_pattern = 'default').tolist():
        if i[1] != []:
            found_default.append(i[1])
    for i in subset['text_eng'].apply(text_cleaning, speakers_pattern = 'expanded').tolist():
        if i[1] != []:
            found_expanded.append(i[1])

    print(mus, found_default, found_expanded, '\n', sep = '\n')

Mamma Mia!
[['SOPHIE / LISA / ALI', 'SOPHIE / ALI / LISA', 'ALI / LISA', 'SOPHIE', 'DONNA', 'LISA', 'ALI'], ['COMPANY', 'PEPPER', 'DONNA', 'TANYA', 'ROSIE', 'ALL'], ['SOPHIE / HARRY / BILL / SAM', 'SOPHIE / HARRY / BILL', 'SOPHIE / HARRY', 'HARRY / SOPHIE', 'SOPHIE', 'HARRY'], ['DONNA / SAM / BILL / HARRY', 'DONNA', 'HARRY', 'BILL', 'SAM'], ['ROSIE', 'DONNA', 'TANYA'], ['DONNA / TANYA / ROSIE', 'DONNA', 'TANYA'], ['SKY / PEPPER / EDDIE / BOYS', 'SOPHIE', 'GIRLS', 'BOYS', 'SKY'], ['DONNA'], ['SOPHIE:', 'SOPHIE', 'HARRY', 'GIRLS', 'BILL', 'SAM'], ['SOPHIE', 'BILL'], ['COMPANY:', 'SOPHIE:', 'HARRY:', 'SAM:'], ['NIGHTMARE CHORUS:', 'SOPHIE:', 'SOPHIE'], ['DONNA:', 'SAM:'], ['DONNA:', 'SAM:'], ['EVERYONE:', 'PEPPER:', 'TANYA:'], ['SAM:'], ['HARRY:', 'DONNA:'], ['DONNA:']]
[['SOPHIE:'], ['COMPANY:', 'SOPHIE:', 'HARRY:', 'Harry:', 'SAM:'], ['NIGHTMARE CHORUS:', 'SOPHIE:'], ['DONNA:', 'SAM:'], ['DONNA:', 'SAM:'], ['EVERYONE:', 'PEPPER:', 'TANYA:'], ['SAM:'], ['HARRY:', 'DONNA:'], ['DONNA:']]



Для всех англ. текстов, кроме "Волос" и "Шахмат", дефолтный лучше подходит. В "Волосах" в принципе нет спикеров, дефолтный паттерн выловил просто кусочки из текста - так что к нему будет expanded, чтобы список оказался пустым. В случае с Шахматами оказалось, что часть поющих обозначена в квадратных скобках в тексте - и они остаются после очистки одним из общих паттернов. 

Эксперимент с подачей кастомного паттерна, который бы ловил и [ПЕРСОНАЖИ А, Б, В], и ПЕРСОНАЖ: одновременно, не удался, поэтому просто потом подчистим "Шахматы" конкретно от текста в квадратных скобках.

In [100]:
songs_filtered[songs_filtered['musical'] == 'Шахматы'].text_eng[2:4]

24    REPORTER 1:\nDoes your opponent\nDeserve such ...
25    [MOLOKOV]\nThe man is utterly mad -- you're pl...
Name: text_eng, dtype: object

In [105]:
patterns_todict_eng = ['default']*(len(mustitles)-1) + ['expanded']
patterndict_eng = dict(zip(mustitles, patterns_todict_eng))
patterndict_eng

{'Mamma Mia!': 'default',
 'Шахматы': 'default',
 'Джекилл и Хайд': 'default',
 'Кошки': 'default',
 'Волосы': 'expanded'}

In [106]:
# теперь в рус. текстах, где ситуация сложнее в целом
speakers_dict = {}
for mus in mustitles:
    subset = songs_filtered[songs_filtered['musical'] == mus]
    # print(mus, len(subset))
    found_default = []
    found_expanded = []
    for i in subset['text_rus'].apply(text_cleaning, speakers_pattern = 'default').tolist():
        if i[1] != []:
            found_default.append(i[1])
    for i in subset['text_rus'].apply(text_cleaning, speakers_pattern = 'expanded').tolist():
        if i[1] != []:
            found_expanded.append(i[1])
    # set_exp = set(found_expanded)
    # set_def = set(found_default)
    
    # diff = set_exp.difference(set_def)
    print(mus, found_default, found_expanded, '\n', sep = '\n')

Mamma Mia!
[]
[]


Шахматы
[['ГОРОЖАНЕ:', 'ФРЕДДИ:', 'МЭР:'], ['ФЛОРЕНС:', 'ФРЕДДИ:', 'ВМЕСТЕ:'], ['ЖУРНАЛИСТЫ:', 'ЖУРНАЛИСТ:', 'ФЛОРЕНС:', 'ФРЕДДИ:'], ['АНАТОЛИЙ:', 'МОЛОКОВ:'], ['АНАТОЛИЙ:'], ['ДИПЛОМАТЫ:', 'МОЛОКОВ:', 'ФЛОРЕНС:'], ['АНСАМБЛЬ:', 'АРБИТР:'], ['АРБИТР:', 'ВСЕ:'], ['АНСАМБЛЬ:', 'МОЛОКОВ:', 'ФРЕДДИ:', 'АРБИТР:', 'ХОР:'], ['СЕРГИЕВСКИЙ:', 'ВСЕ ВМЕСТЕ:', 'ФЛОРЕНС:', 'МОЛОКОВ:', 'АРБИТР:'], ['АНСАМБЛЬ:', 'ФЛОРЕНС:', 'ФРЕДДИ:'], ['АНАТОЛИЙ И ФЛОРЕНС:', 'АНАТОЛИЙ:', 'ФЛОРЕНС:', 'ФРЕДДИ:'], ['ФРЕДДИ И ФЛОРЕНС:', 'ФЛОРЕНС:', 'ФРЕДДИ:'], ['ФРЕДДИ:'], ['АНАТОЛИЙ:', 'СЛУЖАЩИЕ:', 'КЛЕРК:'], ['АНСАМБЛЬ:', 'ФРЕДДИ:'], ['АНСАМБЛЬ:', 'МОЛОКОВ:'], ['АНАТОЛИЙ:', 'СВЕТЛАНА:', 'ФРЕДДИ:'], ['АНАТОЛИЙ:', 'ФЛОРЕНС:'], ['СВЕТЛАНА:', 'ФЛОРЕНС:', 'ВМЕСТЕ:'], ['СВЕТЛАНА:', 'АНСАМБЛЬ:', 'АНАТОЛИЙ:', 'ФЛОРЕНС:', 'ФРЕДДИ:', 'ВСЕ:'], ['СВЕТЛАНА И ФЛОРЕНС:', 'СВЕТЛАНА:', 'АНСАМБЛЬ:', 'АНАТОЛИЙ:', 'ФЛОРЕНС:', 'МОЛОКОВ:', 'АРБИТР:', 'ВСЕ:'], ['АНАТОЛИЙ И ФЛОРЕНС:', 'АНАТОЛИЙ:', 'ФЛОРЕНС:'], ['ВСЕ:', 'И'

В Мамме мии обозначений персонажей в принципе нет, в "Волосах" оно буквально одно (копировалось без них изначально) - так что тут в принципе без разницы. Пусть будет default, он проще. 
Для "Шахмат" и "Кошек" лучше справляется pattern = 'default', а "Джекилл и Хайд" - только expanded.

In [234]:
# создаем словарь с паттернами для русских текстов 
patterns_todict_rus = ['default', 'default', 'expanded', 'default', 'default']
patterndict_rus = dict(zip(mustitles, patterns_todict_rus))
patterndict_rus

{'Mamma Mia!': 'default',
 'Шахматы': 'default',
 'Джекилл и Хайд': 'expanded',
 'Кошки': 'default',
 'Волосы': 'default'}

In [ ]:
# # доп паттерн для Шахмат: r'\[.+\]'

In [236]:
# убираем инфу о говорящих/поющих из всех текстов
# final = songs_filtered[]

for mus in mustitles:
    # subset = songs_filtered[songs_filtered['musical'] == mus]
    mask = songs_filtered['musical'] == mus
    
    # подтягиваем регулярку для этого мюзикла
    eng_pattern = patterndict_eng[mus]
    rus_pattern = patterndict_rus[mus]

    # применяем функцию к датафрейму, но с учетом того, что аргумент с паттерном зависит от значения mus
    clean_eng = songs_filtered.loc[mask, 'text_eng'].apply(
        lambda x: text_cleaning_stringonly(x, speakers_pattern=eng_pattern))
    
    clean_rus = songs_filtered.loc[mask, 'text_rus'].apply(
        lambda x: text_cleaning_stringonly(x, speakers_pattern=rus_pattern))

    # записываем результаты в дф
    songs_filtered.loc[mask, 'clean_eng'] = clean_eng
    songs_filtered.loc[mask, 'clean_rus'] = clean_rus
    # print(clean_rus[:5])

    


In [237]:
# столбец с id пар англ-рус
songs_filtered['id'] = range(len(songs_filtered))
songs_filtered.insert(loc=1, column='id', value=songs_filtered.pop('id'))

C:\Users\Butak\AppData\Local\Temp\ipykernel_10716\689022152.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  songs_filtered['id'] = range(len(songs_filtered))


In [244]:
# дополнительно удалим спикеров в англ. текстах по второму паттерну - для Шахмат, а также для Волос
subset = songs_filtered['musical'] == 'Шахматы'
songs_filtered.loc[subset, 'clean_eng'] = (
    songs_filtered.loc[subset, 'clean_eng'].apply(lambda x: text_cleaning_stringonly(x, speakers_pattern='\[.+\]')))


<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
C:\Users\Butak\AppData\Local\Temp\ipykernel_10716\3702190787.py:4: SyntaxWarning: invalid escape sequence '\['
  songs_filtered.loc[subset, 'clean_eng'].apply(lambda x: text_cleaning_stringonly(x, speakers_pattern='\[.+\]')))


In [245]:
# то же самое для "Волос", как оказалось позж, тоже надо
subset = songs_filtered['musical'] == 'Волосы'
songs_filtered.loc[subset, 'clean_eng'] = (
    songs_filtered.loc[subset, 'clean_eng'].apply(lambda x: text_cleaning_stringonly(x, speakers_pattern='\[.+\]')))

<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
C:\Users\Butak\AppData\Local\Temp\ipykernel_10716\1763455775.py:4: SyntaxWarning: invalid escape sequence '\['
  songs_filtered.loc[subset, 'clean_eng'].apply(lambda x: text_cleaning_stringonly(x, speakers_pattern='\[.+\]')))


In [251]:
final = songs_filtered[['musical', 'id', 'songtitle_eng', 'songtitle_rus', 'clean_eng', 'clean_rus']]
final.head()

,musical,id,songtitle_eng,songtitle_rus,clean_eng,clean_rus
0,Mamma Mia!,0,"Honey, Honey","МИЛЫЙ, МИЛЫЙ","\nHoney honey, how he thrills me, a-ha, honey ...","Милый, Милый,\nОн волнует меня\nМилый, Милый.\..."
1,Mamma Mia!,1,"Money, Money, Money","ДЕНЬГИ, ДЕНЬГИ, ДЕНЬГИ","\nI work all night, I work all day\nTo pay the...","Работай день, работай ночь, чтоб как-то выкрут..."
2,Mamma Mia!,2,Thank You for the Music,СПАСИБО ТЕБЕ ЗА МУЗЫКУ,\nThank you for the music\nThe songs I'm singi...,"Я скажу спасибо тебе за песни,\nИх мелодий хор..."
3,Mamma Mia!,3,Mamma Mia,МАМА МИЯ,\nI was cheated by you\nAnd I think you know w...,"Ты меня обманул, сам ты помнишь когда?\nЯ реши..."
4,Mamma Mia!,4,Chiquitita,ЧИКИТИТА,"\nChiquitita, tell me what's wrong\n\nI have n...","Чикитита, что же с тобой?\nЭто горе так внезап..."


In [252]:
final.to_excel('musdata_final.xlsx')

In [253]:
final.to_csv('musdata_final.csv', encoding='utf-8')

### Обработаем таким же образом еще несколько либретто, которые лежат в другом файле (появились позже)

In [20]:
newsongs = pd.read_excel('addition_libretto_en_ru.xlsx')
newsongs.head(2)

,musical,songtitle_eng,text_eng,songtitle_rus,text_rus
0,Jesus Christ Superstar,Heaven On Their Minds,JUDAS\nMy mind is clearer now.\nAt last all to...,Небом головы полны,"Конец затмения,\nМой разум так чист,\nИ я вижу..."
1,Jesus Christ Superstar,What's The Buzz,APOSTLES\nWhat's the buzz?\nTell me what's a-h...,Что за шум?,"АПОСТОЛЫ:\nЧто за шум, что-нибудь случилось?\n..."


In [21]:
newsongs = newsongs.dropna(subset=['text_eng', 'text_rus'])
mustitles = newsongs['musical'].unique()

In [22]:
# выбор режима обработки пометок о говорящих/поющих

speakers_dict = {}
for mus in mustitles:
    subset = newsongs[newsongs['musical'] == mus]
    
    found_default = []
    found_expanded = []
    for i in subset['text_eng'].apply(text_cleaning, speakers_pattern = 'default').tolist():
        if i[1] != []:
            found_default.append(i[1])
    for i in subset['text_eng'].apply(text_cleaning, speakers_pattern = 'expanded').tolist():
        if i[1] != []:
            found_expanded.append(i[1])

    print(mus,'ENG', found_default, found_expanded, '\n', sep = '\n')
    
    # и для русского

    found_default = []
    found_expanded = []
    for i in subset['text_rus'].apply(text_cleaning, speakers_pattern = 'default').tolist():
        if i[1] != []:
            found_default.append(i[1])
    for i in subset['text_rus'].apply(text_cleaning, speakers_pattern = 'expanded').tolist():
        if i[1] != []:
            found_expanded.append(i[1])

    print('RUS', found_default, found_expanded, '\n', sep = '\n')



Jesus Christ Superstar
ENG
[['JUDAS'], ['MARY MAGDALENE', 'APOSTLES', 'JESUS'], ['ALL (save Judas and Jesus)', 'JUDAS', 'JESUS', 'SIMON'], ['MARY MAGDALENE', 'JUDAS', 'JESUS'], ['MOB (outside)', 'ALL (inside)', 'PRIEST THREE', 'PRIEST TWO', 'PRIEST ONE', 'CAIAPHAS', 'PRIESTS', 'ANNAS'], ['CROWD AND JESUS', 'CROWD (alone)', 'CAIAPHAS', 'JESUS', 'CROWD'], ['SIMON ZEALOTES', 'CROWD'], ['JESUS'], ['PILATE'], ['MONEYCHANGERS AND MERCHANTS', 'JESUS'], ['JESUS', 'CROWD'], ['MARY MAGDALENE'], ['CAIAPHAS', 'JUDAS', 'CHOIR', 'ANNAS'], ['APOSTLES', 'JESUS', 'JUDAS'], ['JESUS'], ['PETER AND APOSTLES', 'CAIAPHAS', 'CROWD', 'JESUS', 'PETER', 'ANNAS'], ['SOLDIER', 'PILATE', 'JESUS', 'MOB'], ['HEROD'], ['CAIAPHAS', 'JUDAS', 'CHOIR', 'ANNAS'], ['CAIAPHAS', 'PILATE', 'JESUS', 'MOB'], ['VOICE OF JUDAS', 'CHOIR']]
[]


RUS
[['МАРИЯ МАГДАЛИНА:', 'АПОСТОЛЫ:', 'ИИСУС:'], ['АПОСТОЛЫ:', 'ИИСУС:', 'ИУДА:'], ['МАГДАЛИНА:', 'ИИСУС:', 'ИУДА:'], ['КАЙАФА:', 'САВЛ:', 'АННА:'], ['ТОЛПА И ИИСУС:', 'КАЙАФА:', 'ТОЛПА:',

In [25]:
newsongs[['musical', 'text_eng', 'text_rus']].tail(3)

,musical,text_eng,text_rus
35,Sound of Music,[Maria:]\nYou are sixteen going on seventeen\n...,"МАРИЯ.\nКоль песня ни разу нес пета,\nСовсем и..."
36,Sound of Music,"CAPTAIN:\nEdelweiss, Edelweiss\nEvery morning ...","ГЕОРГ.\nЭдельвейс, эдельвейс\nЯркий, чистый и ..."
37,Sound of Music,CHORUS:\nA dream that will need\nAll the love ...,АББАТИССА.\nПуть твой за мечтою труден и далёк...


Иисус Христос Суперзвезда - дефолтный. 
Звуки музыки - дефолтный для английского, а в русском нужен будет свой паттерн: пометки идут в верхнем регистре с точкой.

Плюс встречаются и обозначения в [квадратных скобках], которые не очень хорошо вписываются в ту же регулярку. Так что от них надо почистить дополнительно отдельным шагом в функции.

In [19]:
# удаление обозначений со вторым щагом - удаляем все, что внутри квадратных скобок

def v2_text_cleaning_stringonly(song_string, speakers_pattern):
    ''' Удаляет спикеров из текста песни, если они присутствуют '''

    string_copy = song_string
    speakers = get_speakers_by_song(song = song_string, pattern = speakers_pattern)
    if speakers != []:
        for s in speakers:
            string_copy = speakers_deleting(string = string_copy, speakers_set = speakers)

    string_copy = re.sub(r'\[.+?\]', '', string_copy)

    return string_copy

In [87]:
patterns_todict_rus = ['default', 
                       r'[А-ЯЁ \,‒–—―\(\)\\\/\d]+(?: )?(?:\([а-яёА-ЯЁ\, -]+?\))?\.(?: )*?']
patterndict_eng = dict(zip(mustitles, ['default']*2 ))
patterndict_rus = dict(zip(mustitles, patterns_todict_rus))

In [ ]:
# проверка
found_custom_all = []
for mus in mustitles:
    subset = newsongs[newsongs['musical'] == mustitles[1]]
    for i in subset['text_rus'].apply(text_cleaning, speakers_pattern = patterns_todict_rus[1]).tolist():
        if i[1] != []:
            found_custom.append(i[1])

In [88]:
# проверка
found_custom = []
subset = newsongs[newsongs['musical'] == mustitles[1]]
for i in subset['text_rus'].apply(text_cleaning, speakers_pattern = patterns_todict_rus[1]).tolist():
    if i[1] != []:
        found_custom.append(i[1])

In [89]:
found_custom

[['СЕСТРА МАРГАРЕТТА.',
  'СЕСТРА БЕРТА.',
  'СЕСТРА СОФИЯ.',
  'МАРГАРЕТТА.',
  'МАРГЕРЕТТА.',
  'АББАТИССА.',
  'ВМЕСТЕ.',
  'БЕРТА.',
  'СОФИЯ.',
  'ВСЕ.'],
 ['АББАТИССА.', 'ВМЕСТЕ.', 'МАРИЯ.'],
 ['ВМЕСТЕ.', 'МАРИЯ.', 'ДЕТИ.'],
 ['РОЛЬФ.', 'РУЛЬФ.', 'ЛИЗИ.'],
 ['ФРИДРИХ.', 'ГРЕТЛЬ.', 'ВМЕСТЕ.', 'МАРИЯ.', 'ДЕТИ.', 'ЛИЗ.'],
 ['ВМЕСТЕ.', 'ЭЛЬЗА.', 'МАКС.'],
 ['КАПИТАН.', ' МАРИЯ.', 'МАРИЯ.', 'КУРТ.', 'ДЕТИ.', 'ВСЕ.'],
 ['БРИГИТТА.',
  'ФРИДРИХ.',
  'ГРЕТЛЬ.',
  'ЛУИЗА.',
  'МАРТА.',
  'ГОСТИ.',
  'КУРТ.',
  'ДЕТИ.',
  'ЛИЗ.'],
 ['АББАТИССА.'],
 ['КАПИТАН.', ' ЭЛЬЗА.', 'ЭЛЬЗА.', 'МАКС.', 'ВСЕ.'],
 ['КАПИТАН.', 'ВМЕСТЕ.', 'ГЕОРГ.', 'МАРИЯ.'],
 [' ПОСЛУШНИЦЫ.'],
 ['ВМЕСТЕ.', 'МАРИЯ.', 'ЛИЗИ.', 'ЛИЗ.'],
 [' КАПИТАН.', 'КАПИТАН.', ' ДЕТИ.', 'ГЕОРГ.'],
 ['АББАТИССА.', 'ВСЕ.']]

In [57]:
# все в порядке - можно применять
for mus in mustitles:
    mask = newsongs['musical'] == mus
    
    # подтягиваем регулярку для этого мюзикла
    eng_pattern = patterndict_eng[mus]
    rus_pattern = patterndict_rus[mus]

    # применяем функцию к датафрейму, но с учетом того, что аргумент с паттерном зависит от значения mus
    clean_eng = newsongs.loc[mask, 'text_eng'].apply(
        lambda x: v2_text_cleaning_stringonly(x, speakers_pattern=eng_pattern))
    
    clean_rus = newsongs.loc[mask, 'text_rus'].apply(
        lambda x: v2_text_cleaning_stringonly(x, speakers_pattern=rus_pattern))

    # записываем результаты в дф
    newsongs.loc[mask, 'clean_eng'] = clean_eng
    newsongs.loc[mask, 'clean_rus'] = clean_rus


In [58]:
newsongs[:2]

,musical,id,songtitle_eng,text_eng,songtitle_rus,text_rus,clean_eng,clean_rus
0,Jesus Christ Superstar,0,Heaven On Their Minds,JUDAS\nMy mind is clearer now.\nAt last all to...,Небом головы полны,"Конец затмения,\nМой разум так чист,\nИ я вижу...",\nMy mind is clearer now.\nAt last all too wel...,"Конец затмения,\nМой разум так чист,\nИ я вижу..."
1,Jesus Christ Superstar,1,What's The Buzz,APOSTLES\nWhat's the buzz?\nTell me what's a-h...,Что за шум?,"АПОСТОЛЫ:\nЧто за шум, что-нибудь случилось?\n...",\nWhat's the buzz?\nTell me what's a-happening...,"\nЧто за шум, что-нибудь случилось?\nЧто за шу..."


In [59]:
newsongs[-2:]

,musical,id,songtitle_eng,text_eng,songtitle_rus,text_rus,clean_eng,clean_rus
36,Sound of Music,35,Edelweiss,"CAPTAIN:\nEdelweiss, Edelweiss\nEvery morning ...",Edelweiss,"ГЕОРГ.\nЭдельвейс, эдельвейс\nЯркий, чистый и ...","\nEdelweiss, Edelweiss\nEvery morning you gree...","\nЭдельвейс, эдельвейс\nЯркий, чистый и нежный..."
37,Sound of Music,36,Climb Ev'ry Mountain (Reprise),CHORUS:\nA dream that will need\nAll the love ...,Climb Ev'ry Mountain (Reprise),АББАТИССА.\nПуть твой за мечтою труден и далёк...,\nA dream that will need\nAll the love you can...,"\nПуть твой за мечтою труден и далёк,\nНо если..."


In [60]:
# столбец с id пар англ-рус
newsongs['id'] = range(len(newsongs))
newsongs.insert(loc=1, column='id', value=newsongs.pop('id'))

In [63]:
final2 = newsongs[['musical', 'id', 'songtitle_eng', 'songtitle_rus', 'clean_eng', 'clean_rus']]
final2.head(2)

,musical,id,songtitle_eng,songtitle_rus,clean_eng,clean_rus
0,Jesus Christ Superstar,0,Heaven On Their Minds,Небом головы полны,\nMy mind is clearer now.\nAt last all too wel...,"Конец затмения,\nМой разум так чист,\nИ я вижу..."
1,Jesus Christ Superstar,1,What's The Buzz,Что за шум?,\nWhat's the buzz?\nTell me what's a-happening...,"\nЧто за шум, что-нибудь случилось?\nЧто за шу..."


In [64]:
# сохранение
final2.to_excel('additional_musdata.xlsx')